# 02 – Calibration Analysis

Coverage curves, PIT histogram, CRPS, and residual diagnostics for the
aggregate Gaussian predictive distributions.

Run `01_probabilistic_disaggregation.ipynb` first to train the model
and save `reports/tables/test_aggregate_predictions.csv`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.distributions import Normal


ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path(".").resolve()
REPORTS_TABLES = ROOT / "reports" / "tables"
REPORTS_FIGURES = ROOT / "reports" / "figures"
REPORTS_FIGURES.mkdir(parents=True, exist_ok=True)

In [ ]:
test_pred = pd.read_csv(REPORTS_TABLES / "test_aggregate_predictions.csv")
test_pred.head()

In [ ]:
mean = torch.tensor(test_pred["mean_pred"].values, dtype=torch.float32)
std = torch.tensor(test_pred["std_pred"].values, dtype=torch.float32)
y = torch.tensor(test_pred["y_true"].values, dtype=torch.float32)

dist = Normal(mean, std)

# PIT values
pit = dist.cdf(y).numpy()

# Standardized residuals
z = ((y - mean) / std).numpy()

print("Mean standardized residual:", z.mean())
print("Std standardized residual :", z.std())

plt.figure(figsize=(6, 4))
plt.hist(pit, bins=10, edgecolor="black")
plt.xlabel("PIT value")
plt.ylabel("Count")
plt.title("PIT histogram")
plt.tight_layout()
plt.savefig(REPORTS_FIGURES / "pit_histogram.png", dpi=150)
plt.show()

plt.figure(figsize=(6, 4))
plt.hist(z, bins=10, edgecolor="black")
plt.xlabel("Standardized residual")
plt.ylabel("Count")
plt.title("Standardized residuals")
plt.tight_layout()
plt.savefig(REPORTS_FIGURES / "standardized_residuals.png", dpi=150)
plt.show()

In [ ]:
levels = (0.5, 0.8, 0.9, 0.95)
rows = []

for level in levels:
    alpha = 1.0 - level
    lower = dist.icdf(torch.tensor(alpha / 2))
    upper = dist.icdf(torch.tensor(1.0 - alpha / 2))

    inside = ((y >= lower) & (y <= upper)).float()
    width = upper - lower

    rows.append(
        {
            "nominal_coverage": level,
            "empirical_coverage": inside.mean().item(),
            "avg_interval_width": width.mean().item(),
        }
    )

coverage_df = pd.DataFrame(rows)
print(coverage_df.to_string(index=False))

plt.figure(figsize=(5, 5))
plt.plot(
    coverage_df["nominal_coverage"],
    coverage_df["empirical_coverage"],
    marker="o",
    label="model",
)
plt.plot([0, 1], [0, 1], linestyle="--", label="ideal")
plt.xlabel("Nominal coverage")
plt.ylabel("Empirical coverage")
plt.title("Predictive interval calibration")
plt.legend()
plt.tight_layout()
plt.savefig(REPORTS_FIGURES / "calibration_curve.png", dpi=150)
plt.show()

In [ ]:
# CRPS for Gaussian: CRPS(N(mu,sigma), y) = sigma*(z*(2*Phi(z)-1) + 2*phi(z) - 1/sqrt(pi))
from scipy.stats import norm as sp_norm

mu_np = test_pred["mean_pred"].values
sig_np = test_pred["std_pred"].values
y_np = test_pred["y_true"].values

z_np = (y_np - mu_np) / sig_np
crps = sig_np * (
    z_np * (2 * sp_norm.cdf(z_np) - 1)
    + 2 * sp_norm.pdf(z_np)
    - 1 / np.sqrt(np.pi)
)

print(f"Mean CRPS: {crps.mean():.4f}")
print(f"Mean NLL : {(-sp_norm.logpdf(y_np, loc=mu_np, scale=sig_np)).mean():.4f}")